# 06 — TEMPTED–MEFISTO analysis and plots

**Input:** the two full fitted models, the post-method evaluation, and preprocessing metadata/taxonomy  
**Does:** creates the final comparison figures with FIN, EST, and RUS kept separate  
**Output:** figures and small plot-data tables

Because each method is now fit only once, there is no repeated-fit factor matching, sign alignment, reference-batch selection, or consensus-loading machinery.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

COUNTRIES = ["FIN", "EST", "RUS"]
COLORS = {"FIN": "#0072B2", "EST": "#009E73", "RUS": "#D55E00"}

root = Path(".") if Path("data").exists() else Path("..")
tempted_folder = sorted((root / "data" / "tempted").iterdir())[-1]
mefisto_folder = sorted((root / "data" / "mefisto").iterdir())[-1]
evaluation_folder = sorted((root / "data" / "evaluation").iterdir())[-1]
preprocess_folder = sorted((root / "data" / "preprocessing").iterdir())[-1]

output = root / "data" / "figures" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

tempted = pd.read_csv(tempted_folder / "subject_scores.csv", dtype={"subject_id": str})
mefisto = pd.read_csv(mefisto_folder / "subject_scores.csv", dtype={"subject_id": str})
tempted_loadings = pd.read_csv(tempted_folder / "feature_loadings.csv")
mefisto_loadings = pd.read_csv(mefisto_folder / "feature_loadings.csv")
mefisto_samples = pd.read_csv(mefisto_folder / "sample_factors.csv", dtype={"subject_id": str})
metrics = pd.read_csv(evaluation_folder / "metrics.csv")
predictions = pd.read_csv(evaluation_folder / "predictions.csv")
country_recall = pd.read_csv(evaluation_folder / "per_country_recall.csv")
taxonomy = pd.read_csv(preprocess_folder / "taxonomy.csv")

print("Figures:", output)

In [ ]:
def save(name):
    plt.tight_layout()
    plt.savefig(output / name, dpi=220)
    plt.show()
    plt.close()

## Held-out classifier performance

In [ ]:
balanced = metrics.pivot(index="split", columns="method", values="balanced_accuracy")

plt.plot(balanced.index, balanced["TEMPTED"], marker="o", label="TEMPTED")
plt.plot(balanced.index, balanced["MEFISTO"], marker="o", label="MEFISTO")
plt.xlabel("Split")
plt.ylabel("Balanced accuracy")
plt.title("Held-out country classification")
plt.legend()
save("balanced_accuracy_by_split.png")

plt.boxplot(
    [balanced["TEMPTED"], balanced["MEFISTO"]],
    tick_labels=["TEMPTED", "MEFISTO"],
)
plt.ylabel("Balanced accuracy")
plt.title("Balanced accuracy across 20 splits")
save("balanced_accuracy_boxplot.png")

difference = balanced["TEMPTED"] - balanced["MEFISTO"]
plt.axhline(0, color="black", linewidth=1)
plt.scatter(difference.index, difference)
plt.xlabel("Split")
plt.ylabel("TEMPTED − MEFISTO")
plt.title("Paired balanced-accuracy difference")
save("balanced_accuracy_difference.png")

In [ ]:
for method in ["TEMPTED", "MEFISTO"]:
    rows = predictions[predictions["method"] == method]
    matrix = confusion_matrix(rows["truth"], rows["predicted"], labels=COUNTRIES, normalize="true")

    plt.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    plt.colorbar(label="Fraction")
    plt.xticks(range(3), COUNTRIES)
    plt.yticks(range(3), COUNTRIES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{method} confusion matrix")

    for i in range(3):
        for j in range(3):
            plt.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")

    save(f"{method.lower()}_confusion_matrix.png")

mean_recall = country_recall.groupby(["method", "country"])["recall"].mean().unstack(0).reindex(COUNTRIES)
mean_recall.plot.bar()
plt.ylabel("Mean recall")
plt.title("Recall by country")
save("country_recall.png")

## Full-model latent spaces

In [ ]:
for method, scores, x, y in [
    ("TEMPTED", tempted, "component_1", "component_2"),
    ("MEFISTO", mefisto, "factor_1", "factor_2"),
]:
    for country in COUNTRIES:
        rows = scores["country"] == country
        plt.scatter(scores.loc[rows, x], scores.loc[rows, y],
                    s=28, alpha=0.75, color=COLORS[country], label=country)

    plt.xlabel(x.replace("_", " ").title())
    plt.ylabel(y.replace("_", " ").title())
    plt.title(f"{method} subject latent space")
    plt.legend()
    save(f"{method.lower()}_latent_space.png")

## MEFISTO factors over age

MEFISTO exposes fitted factors for every sample, so their longitudinal relationship with age can be plotted directly. TEMPTED's saved table is subject-level and is not artificially projected back to samples.

In [ ]:
for factor in ["factor_1", "factor_2"]:
    for country in COUNTRIES:
        rows = mefisto_samples["country"] == country
        plt.scatter(
            mefisto_samples.loc[rows, "age"],
            mefisto_samples.loc[rows, factor],
            s=12,
            alpha=0.35,
            color=COLORS[country],
            label=country,
        )
    plt.xlabel("Age at collection (days)")
    plt.ylabel(factor.replace("_", " ").title())
    plt.title(f"MEFISTO {factor.replace('_', ' ')} over age")
    plt.legend()
    save(f"mefisto_{factor}_age.png")

## Genus loadings

In [ ]:
def short_name(name):
    return str(name).split("|")[-1].replace("g__", "")

for method, loadings, column in [
    ("TEMPTED", tempted_loadings, "component_1"),
    ("MEFISTO", mefisto_loadings, "factor_1"),
]:
    table = loadings[["feature_id", column]].copy()
    table["absolute"] = table[column].abs()
    table = table.nlargest(15, "absolute").sort_values(column)
    labels = table["feature_id"].map(short_name)

    plt.barh(labels, table[column])
    plt.xlabel("Loading")
    plt.title(f"{method}: strongest first-dimension genus loadings")
    save(f"{method.lower()}_top_loadings.png")

## Cross-method loading correlations

This is the only component/factor matching step needed now. It simply shows the Pearson correlation between the five TEMPTED loading columns and five MEFISTO loading columns over the genera shared by both fits.

In [ ]:
shared = tempted_loadings.merge(
    mefisto_loadings,
    on="feature_id",
    suffixes=("_tempted", "_mefisto"),
)

correlations = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        correlations[i, j] = np.corrcoef(
            shared[f"component_{i+1}"],
            shared[f"factor_{j+1}"],
        )[0, 1]

plt.imshow(correlations, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(5), [f"F{i}" for i in range(1, 6)])
plt.yticks(range(5), [f"C{i}" for i in range(1, 6)])
plt.xlabel("MEFISTO factor")
plt.ylabel("TEMPTED component")
plt.title("Genus-loading correlations")

for i in range(5):
    for j in range(5):
        plt.text(j, i, f"{correlations[i, j]:.2f}", ha="center", va="center")

save("loading_correlations.png")

pd.DataFrame(
    correlations,
    index=[f"component_{i}" for i in range(1, 6)],
    columns=[f"factor_{i}" for i in range(1, 6)],
).to_csv(output / "loading_correlations.csv")

print("Saved:", output)